# Người cầm vật sắc nhọn · sharp_01: detector YOLO cho dao phóng lợn, mã tấu, dao, kiếm, liềm, rìu

Notebook này train detector vật sắc nhọn gồm 7 lớp theo `class_map.json`:
`dao_phong_lon, ma_tau, dao_ngan, kiem, liem, riu, vat_sac_khac`.
Lớp nào cũng dẫn tới cùng một cảnh báo **"Phát hiện người cầm vật sắc nhọn"**.
Việc ghép vật với người (box vật nằm trong hoặc sát box người, đúng K/N khung liên tiếp) làm ở runtime.

**Cách chạy:**
1. Vào *Runtime → Change runtime type*, chọn **A100** (L4 cũng chạy được, chậm hơn).
2. Vào *Runtime → Run all*. Đến bước mount Google Drive thì cho phép truy cập.
3. Phiên Colab **tự ngắt** khi chạy xong, hoặc khi gặp lỗi (traceback được ghi ra `ERROR.txt` trên Drive).

**Dữ liệu:**

| Nguồn | Nội dung | Ghi chú |
|---|---|---|
| Zenodo Dangerous Items | knife → `dao_ngan`, machete → `ma_tau` | CC BY-NC 4.0; ảnh chỉ có súng bị bỏ |
| Granada knife (HF) | knife | CC BY 4.0 |
| Open Images | Knife, Kitchen knife, Dagger, Sword, Axe, tối đa 1500 ảnh mỗi lớp | Kèm **ảnh âm khó** không có box: gậy bóng chày, ô, điện thoại, chai |
| COCO | knife | Tối đa 800 ảnh |
| Simuletic | dao trong cảnh CCTV tổng hợp | |
| **Dữ liệu Việt Nam đã duyệt** trên Drive | | Xem bên dưới |

**Dữ liệu Việt Nam**, đọc trong `MyDrive/SENTINEL/datasets/sharp_object_vn/`:
- `vn_weapons_all_260_training_inputs_009.zip`: chỉ dùng phần `supervised/` (32 ảnh, 84 box đã duyệt).
  228 ảnh trong `unlabeled/` **không** được dùng vì chưa có box.
- Thư mục **`labelled/`**: thả vào đây các file export mới sau khi gán nhãn xong, dạng zip hoặc thư mục, theo một trong hai định dạng:
  - *CVAT → Export → YOLO 1.1*;
  - Ultralytics YOLO (`images/` + `labels/` + `data.yaml`).

  Tên lớp phải trùng với `class_map.json`. Ảnh có file nhãn rỗng bị bỏ qua.
  Nếu một ảnh xuất hiện ở nhiều nguồn, bản mới nhất được dùng.
  Nên kèm một `manifest.csv` có cột `image` / `source_group`, để các ảnh cùng một vụ việc nằm chung một tập.
- **Không** dùng `vn_weapons_cvat_batch_00{1,2,3}.zip` hay `vn_weapons_social_cvat_batch_00{4,5}.zip`:
  đó là box gán nháp tự động, chưa được người duyệt.

**Cách chia tập:** theo nhóm nguồn (cùng video, bài báo hoặc vụ việc thì nằm trọn trong một tập).
Ảnh Việt Nam có val/test riêng, và ảnh train Việt Nam được lặp ×6 trong danh sách train.

**Các bước:**

| Bước | Việc làm | Thời gian ước tính (A100) |
|---|---|---|
| 1 | Tải bộ công khai (chỉ lần đầu; lần sau đọc bản cache trên Drive) | 30–60 phút |
| 2 | Ghép với dữ liệu VN, chia train/val/test | 2–5 phút |
| 3 | Fine-tune YOLO11m ở 960 px, tối đa 100 epoch, dừng sớm sau 25 epoch không cải thiện. Checkpoint ghi ra Drive mỗi epoch; chạy lại notebook thì **train tiếp** từ `last.pt` | 1.5–4 giờ |
| 4 | Đánh giá mAP từng lớp và mAP không phân lớp trên toàn bộ test và trên phần VN. Chọn ngưỡng cảnh báo sao cho tỷ lệ báo nhầm trên ảnh âm của val ≤ 3% | ~5 phút |
| 5 | Đóng gói `sharp_object_<RUN_TAG>.zip` (trọng số fp16, spec, kết quả đánh giá) | 1 phút |

**Kết quả trên Drive:** `MyDrive/SENTINEL/artifacts/sharp_object/<RUN_TAG>/`

**Lưu ý:**
- Val/test VN hiện chỉ có vài ảnh, nên điểm trên phần VN chỉ mang tính tham khảo cho tới khi gán thêm nhãn.
- Quyền sử dụng ảnh báo chí, YouTube và Telegram chưa được xác minh. Chỉ dùng cho đồ án / nghiên cứu.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile, time, traceback, urllib.request, zipfile
from google.colab import drive

RUN_TAG = 'sharp_01'          # đổi tên khi chạy thí nghiệm mới
MODEL = 'yolo11m.pt'          # yolo11s.pt nhanh hơn; yolo11l.pt chính xác hơn nhưng chậm
IMGSZ = 960                   # ảnh lớn giúp thấy lưỡi dao nhỏ
EPOCHS = 100
PUBLIC_POOL_VERSION = 'v1'    # đổi để tải lại bộ công khai
REFRESH_PUBLIC_POOL = False   # True: bỏ cache trên Drive, tải lại bộ công khai
AUTO_DISCONNECT = True        # tự ngắt phiên khi xong hoặc khi lỗi
SOURCE_REVISION = 'e083d67'

assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
roots = [p for p in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')) if p.is_dir()]
assert roots, 'Không thấy MyDrive sau khi mount Google Drive'
SENTINEL = roots[0] / 'SENTINEL'
VN_DIR = SENTINEL / 'datasets' / 'sharp_object_vn'
PUBLIC_CACHE = SENTINEL / 'datasets' / 'sharp_object_public' / f'pool_{PUBLIC_POOL_VERSION}.tar'
OUT = SENTINEL / 'artifacts' / 'sharp_object' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
(VN_DIR / 'labelled').mkdir(parents=True, exist_ok=True)
(OUT / 'drive_write_probe.txt').write_text('write OK', encoding='utf-8')
WORK = Path('/content/sharp_work')
SRC, POOL, DATA, SCRATCH = WORK / 'src', WORK / 'pool', WORK / 'dataset', WORK / 'scratch'
for path in (WORK, SCRATCH):
    path.mkdir(parents=True, exist_ok=True)
STARTED = time.time()


def disconnect(reason):
    print(reason, flush=True)
    if AUTO_DISCONNECT:
        print('Phiên Colab sẽ tự ngắt sau 20 giây (đặt AUTO_DISCONNECT = False để giữ lại).', flush=True)
        time.sleep(20)  # để Drive kịp ghi xong
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()


def on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    text = ''.join(traceback.format_exception(etype, evalue, tb))
    (OUT / 'ERROR.txt').write_text(time.strftime('%Y-%m-%d %H:%M:%S ') + text, encoding='utf-8')
    disconnect(f'LỖI, xem {OUT / "ERROR.txt"} và log trên Drive.')


get_ipython().set_custom_exc((Exception,), on_error)  # lỗi ở bất kỳ cell nào -> ghi Drive + tự ngắt
(OUT / 'ERROR.txt').unlink(missing_ok=True)
print('Kết quả sẽ lưu tại:', OUT)
print('Dữ liệu VN đọc từ:', VN_DIR)

In [ ]:
# Mã nguồn được ghim theo commit để kết quả tái lập được.
marker = SRC / '.revision'
if not marker.is_file() or marker.read_text().strip() != SOURCE_REVISION:
    shutil.rmtree(SRC, ignore_errors=True)
    archive = WORK / 'source.zip'
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=300) as response, archive.open('wb') as output:
        shutil.copyfileobj(response, output)
    with zipfile.ZipFile(archive) as package:
        top = package.namelist()[0].split('/')[0]
        package.extractall(WORK)
    (WORK / top).rename(SRC)
    marker.write_text(SOURCE_REVISION)
print('Mã nguồn:', SOURCE_REVISION)


def run(args, log_name, cwd=SRC):
    """Chạy lệnh, hiện log trực tiếp và lưu log ra Drive."""
    started = time.time()
    with (OUT / log_name).open('a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(map(str, args))}\n')
        process = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        code = process.wait()
    print(f'--- {log_name}: exit {code}, {(time.time() - started) / 60:.1f} phút')
    if code:
        raise RuntimeError(f'{log_name} lỗi (exit {code}), xem {OUT / log_name}')


def py(module, *args, log):
    run([sys.executable, '-u', '-m', module, *args], log)

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.183', 'hf_xet'], 'install.log')
import torch, ultralytics
assert torch.cuda.is_available(), 'Chọn GPU (A100/L4) trong Runtime → Change runtime type'
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

## Bước 1 · Bộ dữ liệu công khai (đã có box)

Chỉ chạy ở lần đầu. Pool được đóng gói thành `pool_<version>.tar` trên Drive, lần sau chỉ cần giải nén.
Nguồn nào lỗi mạng thì bỏ qua và in cảnh báo, các nguồn còn lại vẫn chạy.

In [ ]:
if PUBLIC_CACHE.is_file() and not REFRESH_PUBLIC_POOL:
    print('Dùng pool công khai đã cache:', PUBLIC_CACHE, '(%.2f GB)' % (PUBLIC_CACHE.stat().st_size / 1e9))
    shutil.rmtree(POOL, ignore_errors=True)
    with tarfile.open(PUBLIC_CACHE) as bundle:
        bundle.extractall(WORK)
else:
    failed = []
    for source, extra in (('zenodo', []), ('granada', []), ('simuletic', []),
                          ('coco', ['--coco-limit', 800]),
                          ('openimages', ['--oi-per-class', 1500, '--oi-negatives-per-class', 300])):
        try:
            py('extensions.sharp_object.collect', 'labeled', '--root', POOL, '--work', SCRATCH,
               '--sources', source, *extra, log='collect.log')
        except RuntimeError as error:
            failed.append(source)
            print('CẢNH BÁO: bỏ qua nguồn', source, '-', error)
        shutil.rmtree(SCRATCH, ignore_errors=True)
        SCRATCH.mkdir(parents=True, exist_ok=True)
    assert (POOL / 'manifest.csv').is_file(), 'Không tải được nguồn công khai nào'
    PUBLIC_CACHE.parent.mkdir(parents=True, exist_ok=True)
    partial = PUBLIC_CACHE.with_suffix('.partial')
    with tarfile.open(partial, 'w') as bundle:
        bundle.add(POOL, arcname='pool')
    partial.replace(PUBLIC_CACHE)
    print('Đã cache pool lên Drive:', PUBLIC_CACHE, '| nguồn lỗi:', failed or 'không')
print(json.dumps(json.loads((POOL / 'stats.json').read_text()), indent=1, ensure_ascii=False))

## Bước 2 · Ghép dữ liệu VN đã duyệt và chia tập

Thứ tự ưu tiên đầu vào VN (cái sau thay cái trước khi trùng ảnh):
1. `vn_weapons_all_260_training_inputs_009.zip`
2. mọi zip và thư mục trong `labelled/`, sắp theo thời gian sửa (mới nhất cuối cùng)

In [ ]:
vn_inputs = [VN_DIR / 'vn_weapons_all_260_training_inputs_009.zip']
vn_inputs = [p for p in vn_inputs if p.exists()]
vn_inputs += sorted((p for p in (VN_DIR / 'labelled').iterdir() if p.suffix.lower() == '.zip' or p.is_dir()),
                    key=lambda p: p.stat().st_mtime)
print('Đầu vào VN:')
for path in vn_inputs:
    print('  ', path.relative_to(VN_DIR))
local_inputs = []
for index, path in enumerate(vn_inputs):  # chép về ổ phiên để đọc nhanh
    target = WORK / 'vn_inputs' / f'{index:02d}_{path.name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        shutil.copytree(path, target, dirs_exist_ok=True)
    else:
        shutil.copy(path, target)
    local_inputs.append(target)
py('extensions.sharp_object.build_dataset', '--pool', POOL, '--vn', *local_inputs, '--out', DATA,
   '--vn-repeat', 6, log='build_dataset.log')
shutil.copy(DATA / 'summary.json', OUT / 'dataset_summary.json')
shutil.copy(DATA / 'manifest.csv', OUT / 'dataset_manifest.csv')
summary = json.loads((DATA / 'summary.json').read_text())
for split in ('train', 'val', 'test'):
    print(split, '| ảnh:', summary['images'][split], '| VN:', summary[split]['vn_images'],
          '| ảnh âm:', summary[split]['negatives'], '| box theo lớp:', summary[split]['boxes_by_class'])
for warning in summary['warnings']:
    print('CẢNH BÁO:', warning)

## Bước 3 · Fine-tune YOLO

- Khởi đầu từ trọng số COCO; optimizer AdamW, lr0 = 1e-3, cosine LR; tắt mosaic ở 15 epoch cuối.
- `last.pt` và `best.pt` được ghi vào `train/weights/` trên Drive sau mỗi epoch.
- Nếu phiên bị ngắt giữa chừng, chỉ cần *Run all* lại: dữ liệu được dựng lại y hệt và train tiếp từ `last.pt`.

In [ ]:
batch = 16 if GPU_MEMORY > 30e9 else 8
py('extensions.sharp_object.train_sharp_object', '--stage', 'train', '--data', DATA, '--output', OUT,
   '--model', MODEL, '--imgsz', IMGSZ, '--epochs', EPOCHS, '--batch', batch, '--patience', 25,
   '--workers', 8, log='train.log')

## Bước 4 · Đánh giá và chọn ngưỡng cảnh báo

- **Box mAP**: theo từng lớp, và mAP không phân lớp (vì mọi lớp đều dẫn tới cùng một cảnh báo), trên toàn bộ val/test và trên phần VN.
- **Cảnh báo theo ảnh**: một ảnh bị coi là có cảnh báo khi có box ≥ ngưỡng.
  Ngưỡng là giá trị thấp nhất mà tỷ lệ báo nhầm trên ảnh âm của val ≤ 3%.
  Với ngưỡng đó, đo recall và tỷ lệ báo nhầm trên test.

In [ ]:
py('extensions.sharp_object.train_sharp_object', '--stage', 'evaluate', '--data', DATA, '--output', OUT,
   '--imgsz', IMGSZ, '--max-false-alarm', 0.03, log='evaluate.log')

## Bước 5 · Đóng gói

In [ ]:
py('extensions.sharp_object.train_sharp_object', '--stage', 'export', '--data', DATA, '--output', OUT,
   '--imgsz', IMGSZ, '--run-tag', RUN_TAG, log='export.log')
report = json.loads((OUT / 'evaluation.json').read_text())
fmt = lambda d: {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items() if k != 'per_class'}
for key in ('box_all_test_agnostic', 'box_all_test', 'box_vn_test_agnostic', 'box_vn_val_agnostic'):
    print(f'{key:24s}', fmt(report[key]))
print('\nAP50 test theo lớp:', {k: round(v['ap50'], 3) for k, v in report['box_all_test'].get('per_class', {}).items()})
print('\nNgưỡng cảnh báo:', report['alert_threshold'], '-', report['threshold_rule'])
for key in ('alert_val', 'alert_test', 'alert_vn_test'):
    print(f'{key:16s}', fmt(report[key]))
print('\nFile trên Drive:')
for path in sorted(OUT.rglob('*')):
    if path.is_file() and path.suffix in {'.zip', '.json', '.pt', '.log', '.csv', '.txt'}:
        print(f'  {str(path.relative_to(OUT)):48s} {path.stat().st_size / 1e6:8.2f} MB')
(OUT / 'DONE.txt').write_text(f'xong {time.strftime("%Y-%m-%d %H:%M:%S")}, {(time.time() - STARTED) / 3600:.1f} giờ',
                              encoding='utf-8')
print('\nTổng thời gian: %.1f giờ' % ((time.time() - STARTED) / 3600))
print('Bước tiếp theo: tải sharp_object_' + RUN_TAG + '.zip về máy để đo trên video thật và tích hợp runtime.')

## Tự ngắt phiên

Cell cuối ngắt phiên Colab để không tốn compute unit, vì kết quả đã nằm trên Drive.

In [ ]:
disconnect('XONG. Kết quả ở ' + str(OUT))